# lengua-ligera: el molde en TPU
Entrena en float (JAX) y guarda `.llg`. En casa, Rust lo funde a `.lli` (motor 100% entero).

**Requisitos:** Acelerador = TPU, Internet = On, dataset `lengua-ligera-base` agregado (vocab.txt, valid.bin, molde.py).

In [ ]:
import glob, os
src = os.path.dirname(glob.glob('/kaggle/input/**/molde.py', recursive=True)[0])
os.makedirs('/kaggle/working/data', exist_ok=True)
!cp {src}/vocab.txt {src}/valid.bin /kaggle/working/data/ && cp {src}/molde.py /kaggle/working/
!python -c "import jax; print(jax.__version__, jax.devices())"

## 1. Corpus (~1.5 mil millones de caracteres, todos los shards menos el 10)
Tarda unos 15-30 min. Sólo la primera vez por sesión.

In [ ]:
!cd /kaggle/working && DATA=/kaggle/working/data TRAIN_CHARS=1500000000 python molde.py corpus

## 2. Prueba de velocidad (200 pasos). Copia estas líneas a Claude.

In [ ]:
!cd /kaggle/working && DATA=/kaggle/working/data D=512 L=8 H=8 CTX=256 BATCH=256 STEPS=200 WARMUP=50 EVAL_EVERY=200 EVAL_CHARS=200000 OUT=/kaggle/working/prueba.llg python molde.py train

## 3. Molde float (d512 L8, ~25M parámetros)

In [ ]:
!cd /kaggle/working && DATA=/kaggle/working/data D=512 L=8 H=8 CTX=256 BATCH=256 STEPS=20000 LR=1e-3 WARMUP=500 EVAL_EVERY=2000 OUT=/kaggle/working/float_d512.llg python molde.py train 2>&1 | tee /kaggle/working/float_d512.log

## 4. QAT ternario desde el float

In [ ]:
!cd /kaggle/working && DATA=/kaggle/working/data INIT=/kaggle/working/float_d512.llg QAT=ternario-g32 STEPS=6000 LR=1e-3 WARMUP=100 EVAL_EVERY=1000 OUT=/kaggle/working/tern_d512.llg python molde.py train 2>&1 | tee /kaggle/working/tern_d512.log

## 5. Control justo: float con los mismos pasos extra (sin cuantizar)

In [ ]:
!cd /kaggle/working && DATA=/kaggle/working/data INIT=/kaggle/working/float_d512.llg STEPS=6000 LR=1e-3 WARMUP=100 EVAL_EVERY=1000 OUT=/kaggle/working/ctrl_d512.llg python molde.py train 2>&1 | tee /kaggle/working/ctrl_d512.log

Descarga `*.llg` y `*.log` desde la pestaña Output (o botón *Save Version* → Output).